In [48]:
# Import necessary libraries
import os
from dotenv import load_dotenv
import logging

from llama_index.llms.openai import OpenAI
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader
from llama_index.core.node_parser import SimpleNodeParser
from llama_index.core.utils import get_tokenizer
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

In [45]:
# Get API key from environment variable
load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

# Validate that the API key is set
if not OPENAI_API_KEY:
    raise ValueError("OPENAI_API_KEY is not set in the environment variables.")

In [25]:
# Function to analyze paragraph lengths in the documents 
# and suggest the optimal chunking parameters
def analyze_paragraph_lengths(doc_dir):
    # Load docs
    documents = SimpleDirectoryReader(doc_dir).load_data()
    tokenizer = get_tokenizer()

    paragraphs = []
    for doc in documents:
        # Split on double newline as paragraph separator
        for p in doc.text.split("\n\n"):
            p = p.strip()
            if not p:
                continue
            paragraphs.append(p)

    paragraph_lengths = [len(tokenizer(p)) for p in paragraphs]
    if not paragraph_lengths:
        print("No paragraphs found.")
        return None

    par_cnt = len(paragraph_lengths)
    avg_len = sum(paragraph_lengths) / len(paragraph_lengths)
    max_len = max(paragraph_lengths)
    min_len = min(paragraph_lengths)

    # Heuristic: chunk_size ≈ 1.5 × avg paragraph length, but at least 200
    chunk_size = max(200, int(avg_len * 1.5))
    # Overlap ≈ 15% of chunk_size
    overlap = int(chunk_size * 0.15)

    return {
        "paragraph_count": par_cnt,
        "paragraph_lengths": paragraph_lengths,
        "average_paragraph_length": avg_len,
        "chunk_size": chunk_size,
        "chunk_overlap": overlap,
    }

In [26]:

token_params = analyze_paragraph_lengths("pdf")

for key, value in token_params.items():
    print(f"{key}: {value}")

paragraph_count: 5
paragraph_lengths: [465, 471, 407, 260, 329]
average_paragraph_length: 386.4
chunk_size: 579
chunk_overlap: 86


In [46]:
# Instantiate the tokenizer to use in the node parser
tokenizer = get_tokenizer()

# Load documents from the specified directory
documents = SimpleDirectoryReader("pdf").load_data()

# Parse the loaded documents into nodes
# The function above suggests using chunk_size=579 and chunk_overlap=86 
# based on the analysis of paragraph lengths. However, looking at the 
# length of each paragraph, we can see that all paragraphs are with less 
# than 579 tokens. Hence, we will use chunk_size=400 and chunk_overlap=60.
parser = SimpleNodeParser(
    chunk_size=400, 
    chunk_overlap=60, 
    separator="\n\n"
)
nodes = parser.get_nodes_from_documents(documents)

# Print the content of the first five nodes to verify the parsing
for i, node in enumerate(nodes[:5]):
    token_count = len(tokenizer(node.text))
    print(f"\n --- Node {i} - Token Count: {token_count} ---\n")
    print(node.text)
    print(f"\n --- End of Node {i} ---\n")



 --- Node 0 - Token Count: 333 ---

What is Retrieval-Augmented Generation? 
Retrieval-Augmented Generation (RAG) is the process of optimizing the output of a large 
language model, so it references an authoritative knowledge base outside of its training 
data sources before generating a response. Large Language Models (LLMs) are trained on 
vast volumes of data and use billions of parameters to generate original output for tasks 
like answering questions, translating languages, and completing sentences. RAG extends 
the already powerful capabilities of LLMs to specific domains or an organization's internal 
knowledge base, all without the need to retrain the model. It is a cost-effective approach to 
improving LLM output so it remains relevant, accurate, and useful in various contexts. 
Why is Retrieval-Augmented Generation important? 
LLMs are a key artificial intelligence (AI) technology powering intelligent chatbots and 
other natural language processing (NLP) applications. The go

In [ ]:
# ******* Use embedding model as well as the LLM both from OpenAI *******

# Initialize the OpenAI LLM with the API key and model
model = OpenAI(api_key=OPENAI_API_KEY, model="gpt-4o-mini")

# Initialize the OpenAI embedding model
embedding_model = OpenAIEmbedding(
    api_key=OPENAI_API_KEY
    , model="text-embedding-3-small"
)

# Create a in memory vector store index from the nodes 
# and the embedding model
index = VectorStoreIndex(
    nodes,
    embed_model=embedding_model,
    similarity_top_k=3
)

# Instantiate the query engine with the LLM and the index
query_engine = index.as_query_engine(llm=model)

# Example query
question = "Which Amazon service is used for RAG?"

# Get the response from the query engine
response = query_engine.query(question)

# Print the response
print(response)

2026-03-08 20:37:08,744 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-03-08 20:37:09,749 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-03-08 20:37:12,025 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


Amazon Bedrock is used for Retrieval-Augmented Generation (RAG). Additionally, Amazon Kendra serves as a highly-accurate enterprise search service that can be utilized in RAG workflows.


In [50]:
# ******* Use embedding model from HuggingFace and LLM from OpenAI *******

# Note: HuggingFace embedding models can be quite verbose in their logging,
# so we set the logging level to ERROR to suppress unnecessary logs.
logging.getLogger("sentence_transformers").setLevel(logging.ERROR)
logging.getLogger("transformers").setLevel(logging.ERROR)
logging.getLogger("huggingface_hub").setLevel(logging.ERROR)

# Initialize the OpenAI LLM with the API key and model
model = OpenAI(api_key=OPENAI_API_KEY, model="gpt-4o-mini")

# Initialize the Free / local embedding model from HuggingFace
embedding_model = HuggingFaceEmbedding(
    model_name="BAAI/bge-small-en"
)

# Create a in memory vector store index from the nodes 
# and the embedding model
index = VectorStoreIndex(
    nodes,
    embed_model=embedding_model,
    similarity_top_k=3
)

# Instantiate the query engine with the LLM and the index
query_engine = index.as_query_engine(llm=model)

# Example query
question1 = "Which Amazon service is used for RAG?"
question2 = "Why is RAG cost effective?"

# Get the response from the query engine
# response1 = query_engine.query(question1)
response2 = query_engine.query(question2)

# Print the response
# print(response1)
print(response2)

2026-03-08 21:35:43,586 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-03-08 21:35:43,643 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en/2275a7bdee235e9b4f01fa73aa60d3311983cfea/modules.json "HTTP/1.1 200 OK"
2026-03-08 21:35:43,693 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-03-08 21:35:43,709 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en/2275a7bdee235e9b4f01fa73aa60d3311983cfea/config_sentence_transformers.json "HTTP/1.1 200 OK"
2026-03-08 21:35:43,758 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-03-08 21:35:43,774 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/model

RAG is cost-effective because it allows organizations to introduce new data to a foundation model without the high computational and financial costs associated with retraining the model for specific information. This approach makes generative AI technology more accessible and usable, reducing the overall expenses involved in maintaining and updating the model with relevant data.
